# 01 Build Loan-Month Panel

**Purpose:** Join origination traits to the monthly history and flag first default / first prepayment.

**Steps**
- Join on Loan Sequence Number
- Derive vintage quarter and MOB
- Default flag (90+ DPD / RA / loss ZBC), absorbing
- Prepay flag (ZBC 01)
- Loan-level first event table
- Sanity checks

**Read first:** Bellini 2.2 (default definition)

**Done when:** panel and loan-level event table for 2007 saved to `data/processed/`, checks pass.

**Year in this pass:** 2007 only. Functions that work get moved into `src/freddie/`.

## 1. Setup

Load the libraries and this project's own functions from `src/freddie`.

In [1]:
# Imports + Notebook Setup

import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.append(r"C:\Dev\CreditRiskLearning\Projects\Freddie Mac End-to-End Project\src")
from freddie.config import RAW_DIR, INTERIM_DIR, DEFAULT_ZBC, PREPAY_ZBC, CENSOR_ZBC, PROCESSED_DIR

from freddie.load import load_orig, load_perf, sanity_check
from freddie.panel import add_mob, vintage_quarter, default_flagger, default_flag_absorbing, prepay_flag, censor_flag, summarise_loans

years = [2006, 2007, 2010, 2016]

## 2. Load the data

Two files per year. **orig:** one row per loan, what we knew when it was approved. **perf:** one row per loan per month, what happened next.

In [2]:
orig = {year: load_orig(year, save=False) for year in years}

2006 orig: 50,000 rows x 31 cols
2007 orig: 50,000 rows x 31 cols
2010 orig: 50,000 rows x 31 cols
2016 orig: 50,000 rows x 31 cols


In [3]:
perf= {year: load_perf(year, save=False) for year in years}

## 3. Label every month

Each monthly row gets a few labels that the vintage curves will count later.

### 3.1 Vintage

The quarter the loan was written in, read from the loan ID (`F06Q1...` is 2006 Q1).

In [4]:
# Add Vintage Quarter

for year in years: 
    vintage_quarter(perf[year])
    


### 3.2 Months on book (MOB)

How old the loan is in each month. Counted from the calendar, because Freddie resets `loan_age` when a loan is modified.

In [5]:
# Add MOB
# loan_age resets after a loan modification (User Guide, LOAN AGE), so it can't be the clock.
# MOB counts months from the first payment date to the reporting period, +1 so it matches loan_age on unmodified loans.

for year in years: 
    add_mob(perf[year], orig[year])
    


### 3.3 Default

A month counts as a default if the borrower is 90+ days behind, the home has been repossessed, or the loan ended with a loss. Once a loan defaults it stays defaulted, even if the borrower later catches up.

In [ ]:
# Add 90+ DPD Flag
for year in years: 
    default_flagger(perf[year])

In [ ]:
# Default absorbing function
for year in years: 
    perf[year] = default_flag_absorbing(perf[year])

In [8]:
perf[2006].sample(10)

,loan_id,period,current_upb,current_dlq_status,loan_age,zero_balance_code,zero_balance_date,vintage,mob,default_flag,default_flag_absorbing
393436,F06Q10168148,200711,90033.99,00,21,NaN,<NA>,2006Q1,21,0,0
1450301,F06Q20310482,200803,131102.83,00,20,NaN,<NA>,2006Q2,20,0,0
599064,F06Q10255123,200904,224758.69,00,37,NaN,<NA>,2006Q1,37,0,0
458534,F06Q10197604,200710,92242.02,00,18,NaN,<NA>,2006Q1,18,0,0
972119,F06Q20051740,201001,25021.45,00,43,NaN,<NA>,2006Q2,43,0,0
161413,F06Q10068589,200902,336111.12,00,35,NaN,<NA>,2006Q1,35,0,0
2669461,F06Q40145249,200707,77542.95,00,8,NaN,<NA>,2006Q4,8,0,0
1449411,F06Q20309831,201207,103082.14,00,72,NaN,<NA>,2006Q2,72,0,0
3098247,F06Q40367570,200710,206455.81,00,9,NaN,<NA>,2006Q4,9,0,0
556802,F06Q10237437,200910,224693.93,11,17,NaN,<NA>,2006Q1,43,1,1


### 3.4 Prepaid and censored

**Prepaid:** the loan was paid off in full. **Censored:** the loan left the data for an admin reason, so we stop watching it. Neither counts if the loan had already defaulted.

In [9]:
# Add Prepay ZBD Flag
for year in years: 
    prepay_flag(perf[year])
    

In [ ]:
# Censor Flag

for year in years: 
    censor_flag(perf[year])

## 4. One row per loan

Shrink the monthly history to one line per loan: when it was written, what happened to it first, and how many months in.

In [ ]:
# Collapse into Loan Level First Event Table

loan_table = {}

for year in years: 
    loan_table[year] = summarise_loans(perf[year])

## 5. Findings

What the checks turned up, and the decisions made because of it.

**Observation (Written by AI).** 142 loans showed both a prepay and a default. All 142 had a default *after* the payoff row when sorted by `loan_age`. Cause: the User Guide says `loan_age` restarts from the modification first payment date for modified loans. In 2006, 2,651 loans (about 5%) have at least one reset and 133,321 rows share a `loan_age` with another row of the same loan, so sorting by `loan_age` put their months out of time order.

**Decision.** Use `mob` (months from `first_payment_date` to `period`, +1) as the clock for every sort and event month. It never resets and equals `loan_age` on unmodified loans. With `mob`, the overlap is 0 and outcome counts are unchanged (prepaid 41,951, default 7,215, open 744, censored 90), but 1,107 of the 7,215 defaults (15%) move to a different default month (median 26 months later). `loan_age` is kept in the panel for reference only.


## 6. Exports for next stage

In [ ]:
for year in years:
    
    loan_table[year].to_parquet(PROCESSED_DIR / f"loan_table_{year}.parquet", index=False)